In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch

# ============================
# Base model and configuration
# ============================
model_name    = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
SEED          = 69
MAX_SEQ_LENGTH = 1024

# ============================
# Load model and tokenizer for LoRA finetuning
# ============================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = model_name,
    max_seq_length  = MAX_SEQ_LENGTH,   # should match your dataset needs
    load_in_4bit    = True,             # required for 4bit LoRA finetuning
    load_in_8bit    = False,
    full_finetuning = False,            # keep this False for LoRA training
)

# ============================
# Prepare model for LoRA / DoRA finetuning
# ============================
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,    # keep False unless you explicitly want RS LoRA
    use_dora = True,       # enables DoRA
    loftq_config = None,   # no LoftQ quantization config
)

print("Model loaded and ready for LoRA finetuning.")


In [ ]:
import pandas as pd
from datasets import Dataset
from collections import Counter   # for curriculum

# ---------- Paths ----------

OUTPUT_DIR = "/content/drive/MyDrive/qwen3-8b-unsloth-dora-curriculum-learning"

CSV_PATH   = "/content/drive/MyDrive/train.csv"


print(f"Loading dataset from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Check required columns exist
required_cols = {"input_finding", "output_disease"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# Keep required columns and drop rows with missing values
df = df[["input_finding", "output_disease"]].dropna()
df["input_finding"] = df["input_finding"].astype(str).str.strip()
df["output_disease"] = df["output_disease"].astype(str).str.strip()

print("Number of rows after filtering:", len(df))
print("Columns:", list(df.columns))
display(df.head())

# ---------- 1. Build allowed disease label list ----------
disease_labels_series = (
    df["output_disease"]
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .dropna()
)

# Filter out any empty strings after stripping
disease_labels_series = disease_labels_series[disease_labels_series != ""]

disease_labels = sorted(disease_labels_series.unique().tolist())
allowed_labels = ", ".join(disease_labels)

print("Number of unique disease labels:", len(disease_labels))
preview = allowed_labels[:300]
print("Allowed labels preview:", preview, "..." if len(allowed_labels) > 300 else "")

# ---------- 2. Curriculum based on label frequency ----------

def split_labels(s):
    return [x.strip() for x in str(s).split(",") if x.strip()]

# Compute global label counts
all_labels = df["output_disease"].apply(split_labels)
flat_labels = [lab for labs in all_labels for lab in labs]
label_counts = Counter(flat_labels)

print("Top 10 most frequent labels:")
print(pd.Series(label_counts).sort_values(ascending=False).head(10))

def avg_label_freq(label_str):
    labels = split_labels(label_str)
    if not labels:
        return 0.0
    counts = [label_counts[lab] for lab in labels]
    return sum(counts) / len(counts)

# Per sample difficulty proxy: higher avg_label_freq = easier (more common labels)
df["avg_label_freq"] = df["output_disease"].apply(avg_label_freq)

# Sort by difficulty: high frequency first
df_sorted = df.sort_values("avg_label_freq", ascending=False).reset_index(drop=True)

# Three curriculum phases: easiest third, easiest two thirds, full set
phase1_frac = 1.0 / 3.0      # about 33 percent
phase2_frac = 2.0 / 3.0      # about 67 percent

n_phase1 = int(phase1_frac * len(df_sorted))
n_phase2 = int(phase2_frac * len(df_sorted))

df_phase1 = df_sorted.iloc[:n_phase1].copy()          # easiest, highest frequency labels
df_phase2 = df_sorted.iloc[:n_phase2].copy()          # easy plus medium
df_phase3 = df_sorted.copy()                          # full dataset

print(f"Phase 1 size (easiest third): {len(df_phase1)}")
print(f"Phase 2 size (easiest two thirds): {len(df_phase2)}")
print(f"Phase 3 size (full dataset): {len(df_phase3)}")

# ---------- 3. Convert to Hugging Face Datasets ----------

hf_dataset_phase1 = Dataset.from_pandas(
    df_phase1[["input_finding", "output_disease"]],
    preserve_index=False,
)

hf_dataset_phase2 = Dataset.from_pandas(
    df_phase2[["input_finding", "output_disease"]],
    preserve_index=False,
)

hf_dataset_phase3 = Dataset.from_pandas(
    df_phase3[["input_finding", "output_disease"]],
    preserve_index=False,
)

print("Phase 1 dataset:", hf_dataset_phase1)
print("Phase 2 dataset:", hf_dataset_phase2)
print("Phase 3 dataset:", hf_dataset_phase3)


In [ ]:
import numpy as np

# System prompt used in every training example
system_prompt = (
    "You are a clinical Named Entity Recognition (NER) and multi-label classification model. "
    "Read the abdominal radiology findings and identify all diseases that are present. "
    "Use only disease names from the allowed disease label list. "
    "Return the diseases as a comma separated list using the exact wording from the list. "
    "If none apply, output: No acute abnormality"
    f"Allowed disease label list: {allowed_labels}."
)

def format_batch(batch):
    texts = []
    for finding, labels in zip(batch["input_finding"], batch["output_disease"]):
        finding = str(finding).strip()
        labels = str(labels).strip()

        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    "Clinical findings:\n"
                    f"{finding}\n\n"
                    "List all diseases present using only labels from the allowed disease list. "
                    "Separate multiple diseases with commas. If none apply, output: No acute abnormality"
                ),
            },
            {"role": "assistant", "content": labels},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)

    return {"text": texts}

print("Applying chat template to curriculum datasets...")

# IMPORTANT CHANGE: apply to phase 1, 2, 3 instead of hf_dataset

processed_phase1 = hf_dataset_phase1.map(
    format_batch,
    batched=True,
    remove_columns=hf_dataset_phase1.column_names,
)

processed_phase2 = hf_dataset_phase2.map(
    format_batch,
    batched=True,
    remove_columns=hf_dataset_phase2.column_names,
)

processed_phase3 = hf_dataset_phase3.map(
    format_batch,
    batched=True,
    remove_columns=hf_dataset_phase3.column_names,
)

print("Example processed text (phase 1):")
example_text = processed_phase1[0]["text"]
print(example_text)

# -------------------------------
# Compute token length statistics on final phase (full data)
# -------------------------------
lengths = []

for sample in processed_phase3["text"]:
    encoded = tokenizer(
        sample,
        add_special_tokens=False,
    )
    lengths.append(len(encoded["input_ids"]))

lengths = np.array(lengths)

print("\nToken length statistics for training texts (phase 3, full set):")
print("Number of samples:", len(lengths))
print("Min length:", int(lengths.min()))
print("Max length:", int(lengths.max()))
print("Mean length:", float(lengths.mean()))
print("Median length (50th percentile):", int(np.percentile(lengths, 50)))
print("90th percentile:", int(np.percentile(lengths, 90)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("99th percentile:", int(np.percentile(lengths, 99)))

example_len = len(tokenizer(example_text, add_special_tokens=False)["input_ids"])
print("\nToken length of example 0:", example_len)


In [ ]:
import os
from trl import SFTTrainer, SFTConfig

# Disable wandb globally (Colab safe)
os.environ["WANDB_DISABLED"] = "true"

# Training hyperparameters
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
EPOCHS         = 4
LR             = 2e-4

print("Setting up training configuration...")

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text",
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs = EPOCHS,
    learning_rate = LR,
    warmup_steps = 50,
    logging_steps = 10,
    save_strategy = "epoch",
    output_dir = OUTPUT_DIR,
    optim = "adamw_8bit",
    bf16 = use_bf16,
    fp16 = not use_bf16,
    seed = SEED,

    # Disable wandb here too
    report_to = "none",
)

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
phase1_dir = os.path.join(OUTPUT_DIR, "phase1")
phase2_dir = os.path.join(OUTPUT_DIR, "phase2")
phase3_dir = os.path.join(OUTPUT_DIR, "phase3")
os.makedirs(phase1_dir, exist_ok=True)
os.makedirs(phase2_dir, exist_ok=True)
os.makedirs(phase3_dir, exist_ok=True)

# -----------------------------
# PHASE 1  easiest third
# -----------------------------
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = processed_phase1,
    args          = training_args,
    packing       = True,
)

print("Starting Phase 1 training...")
trainer.train()
trainer.save_model(phase1_dir)
print("Phase 1 complete. Saved to:", phase1_dir)

# -----------------------------
# PHASE 2  easiest two thirds
# -----------------------------
trainer = SFTTrainer(
    model         = model,             # continues from phase 1 weights
    tokenizer     = tokenizer,
    train_dataset = processed_phase2,
    args          = training_args,
    packing       = True,
)

print("Starting Phase 2 training...")
trainer.train()
trainer.save_model(phase2_dir)
print("Phase 2 complete. Saved to:", phase2_dir)

# -----------------------------
# PHASE 3  full dataset
# -----------------------------
trainer = SFTTrainer(
    model         = model,             # continues from phase 2 weights
    tokenizer     = tokenizer,
    train_dataset = processed_phase3,
    args          = training_args,
    packing       = True,
)

print("Starting Phase 3 training...")
trainer.train()
trainer.save_model(phase3_dir)
print("Phase 3 complete. Saved to:", phase3_dir)
